<a href="https://colab.research.google.com/github/whgusdn5221/comfycolab/blob/main/sdxl_v1.0_comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, atexit, requests, subprocess, time, re
from random import randint
from threading import Timer
from queue import Queue
from google.colab import drive


# 1. 구글 드라이브 마운트 (권한 승인 팝업이 뜹니다)
drive.mount('/content/drive')

# 2. 드라이브 내 모델 저장 경로 설정
drive_model_path = "/content/drive/MyDrive/ComfyUI/models/checkpoints"
os.makedirs(drive_model_path, exist_ok=True)

# 3. 코랩 로컬 폴더와 드라이브 폴더 연결 (심볼릭 링크)
!rm -rf /content/ComfyUI/models/checkpoints
!ln -s {drive_model_path} /content/ComfyUI/models/checkpoints

# 4. 모델 다운로드 함수 (파일이 없을 때만 실행)
def download_model(url, file_name, api_token=""):
    target_path = os.path.join(drive_model_path, file_name)
    if not os.path.exists(target_path):
        print(f"🚀 {file_name} 다운로드 시작...")
        if "civitai.com" in url:
            !wget -O {target_path} "{url}?token={api_token}"
        else:
            !wget -O {target_path} {url}
    else:
        print(f"✅ {file_name} 이미 드라이브에 있습니다. 스킵합니다!")

# --- 모델 리스트 실행 ---
# Flux.2는 용량이 매우 크니 드라이브 용량 확인 필수!
download_model("https://huggingface.co/black-forest-labs/FLUX.2-klein-9B/resolve/main/flux2-klein-9b.safetensors", "flux2-klein-9b.safetensors")
# Z-Image나 Pony 모델도 같은 방식으로 추가 (API 토큰 필요)
# download_model("Civitai주소", "z_image.safetensors", "YOUR_API_TOKEN")

# [1] 시스템 최적화 및 필수 라이브러리 설치 (한 번에 실행)
!apt -y update -qq && apt -y install -qq aria2
!wget https://github.com/camenduru/gperftools/releases/download/v1.0/libtcmalloc_minimal.so.4 -O /content/libtcmalloc_minimal.so.4
%env LD_PRELOAD=/content/libtcmalloc_minimal.so.4

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers triton mediapipe addict yapf fvcore omegaconf gitpython insightface onnxruntime-gpu

# [2] ComfyUI 및 필수 노드 클린 설치
%cd /content
if not os.path.exists('ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !git clone https://github.com/ltdrdata/ComfyUI-Manager custom_nodes/ComfyUI-Manager
    !git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus custom_nodes/ComfyUI_IPAdapter_plus
else:
    %cd /content/ComfyUI

# [3] 구글 드라이브 마운트 및 모델 자동 정렬 (핵심 최적화)
drive.mount('/content/drive')

# 드라이브 내 모델 경로 및 파일명 자동 정규화
model_types = ["checkpoints", "clip_vision", "ipadapter", "vae", "loras", "upscale_models"]
for m_type in model_types:
    drive_path = f"/content/drive/MyDrive/ComfyUI/models/{m_type}"
    colab_path = f"/content/ComfyUI/models/{m_type}"

    if os.path.exists(drive_path):
        # 1. ClipVision 파일 이름 강제 표준화 (Unified Loader 에러 방지)
        if m_type == "clip_vision":
            old_f = f"{drive_path}/clip_vision_vit_h.safetensors"
            new_f = f"{drive_path}/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors"
            if os.path.exists(old_f):
                os.rename(old_f, new_f)
                print(f"✅ ClipVision 이름 표준화 완료")

        # 2. 심볼릭 링크 생성
        !rm -rf {colab_path}
        !ln -s {drive_path} {colab_path}
        print(f"✅ {m_type} 폴더 연결 완료")

# [4] Cloudflare 터널 생성 및 주소 출력
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64

def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        time.sleep(3)
        try:
            tunnel_url = re.search(r"(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            attempts += 1
    output_queue.put(tunnel_url if tunnel_url else "❌ 터널 생성 실패")

output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
print(f"\n🚀 접속 주소: {output_queue.get()}\n")

# [5] ComfyUI 가동
!python main.py --dont-print-server

In [ ]:
# ComfyUI 출력 폴더 안의 모든 파일 삭제 (폴더 구조는 유지)
!find /content/ComfyUI/output -type f -delete
print("✅ output 폴더가 깨끗하게 비워졌습니다!")

In [ ]:
# 1. 모델 저장 폴더로 이동 (ComfyUI 기준)
%cd /content/ComfyUI/models/checkpoints

# --- [1번 모델] Flux.2 klein-9B (HuggingFace) ---
# Flux 모델은 용량이 크므로 다운로드에 시간이 걸릴 수 있습니다.
!wget -O flux2-klein-9b.safetensors https://huggingface.co/black-forest-labs/FLUX.2-klein-9B/resolve/main/flux2-klein-9b.safetensors

# --- [2번 모델] Z-Image Turbo NSFW (Civitai) ---
# 모델 ID는 예시입니다. 실제 페이지의 Download 링크 숫자를 확인하세요.
!wget -O z_image_turbo_nsfw.safetensors "https://civitai.com/api/download/models/123456?token=YOUR_CIVITAI_API_KEY"

# --- [3번 모델] Pony Realism (Civitai) ---
!wget -O pony_realism_v2.safetensors "https://civitai.com/api/download/models/987654?token=YOUR_CIVITAI_API_KEY"

# 다운로드 완료 확인
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')